# YOLO26 × KITTI (grayscale) — Colab 실행 노트북

**시작 전에 한 번만**
1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. Google 드라이브 `내 드라이브`(MyDrive) 맨 위에 아래 3개 파일을 올려 두기 (이름 그대로, 압축 풀지 말 것)
   - `yolo26-kitti.zip` (코드)
   - `data_object_image_2.zip` (KITTI 이미지, 12GB)
   - `data_object_label_2.zip` (KITTI 라벨, 5MB)

그다음 위에서부터 셀을 하나씩 실행(▶ 또는 Shift+Enter)하면 돼요.
연결이 끊기면 1번 셀부터 다시 실행하세요. 데이터 준비는 건너뛰고, 학습은 끊긴 epoch부터 이어서 해요.

In [ ]:
# 1. GPU 확인 + 드라이브 연결
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 설정 (파일 이름이나 개수를 바꿨다면 여기만 고치세요)
import os
DRIVE = '/content/drive/MyDrive'
CODE_ZIP = f'{DRIVE}/yolo26-kitti.zip'
KITTI_IMG_ZIP = f'{DRIVE}/data_object_image_2.zip'
KITTI_LABEL_ZIP = f'{DRIVE}/data_object_label_2.zip'
SUBSET_ZIP = f'{DRIVE}/kitti_yolo.zip'      # 3번 셀이 만들어 저장하는 3,000장 데이터
RUNS_DIR = f'{DRIVE}/yolo26_runs'          # 학습 결과가 바로 저장되는 곳
NUM_IMAGES = 3000
DATA = '/content/kitti_yolo'
REPO = '/content/yolo26-kitti'

for f in (CODE_ZIP, KITTI_LABEL_ZIP):
    print(('OK   ' if os.path.exists(f) else '없음 ') + f)
print(('OK   ' if os.path.exists(KITTI_IMG_ZIP) or os.path.exists(SUBSET_ZIP) else '없음 ') + KITTI_IMG_ZIP)

In [ ]:
# 3. 코드 풀기 + 데이터 준비
#    처음 한 번: 드라이브 zip -> Colab 디스크로 복사 -> testing 폴더는 빼고 압축 해제 -> 3,000장 변환 -> kitti_yolo.zip을 드라이브에 저장 (30분 안팎)
#    두 번째부터: 드라이브의 kitti_yolo.zip만 풀어서 1~2분
if not os.path.exists(CODE_ZIP):
    raise FileNotFoundError(f'코드 zip이 없어요: {CODE_ZIP}\n드라이브 "내 드라이브" 맨 위에 yolo26-kitti.zip(이름 정확히)이 있는지 확인하세요.')
!rm -rf {REPO} /content/_code && unzip -q -o "{CODE_ZIP}" -d /content/_code
import glob, shutil
found = sorted(glob.glob('/content/_code/**/train.py', recursive=True), key=len)
if not found:
    raise FileNotFoundError('zip 안에 train.py가 없어요. 받은 yolo26-kitti.zip 원본을 다시 올려 주세요.')
shutil.move(os.path.dirname(found[0]), REPO)   # 폴더가 한 번 더 감싸져 있어도 OK
!rm -rf /content/_code
print('코드 OK:', REPO)
if os.path.exists(SUBSET_ZIP):
    print('저장된 3,000장 데이터를 사용해요')
    !cp {SUBSET_ZIP} /content/ && unzip -q -o /content/kitti_yolo.zip -d /content/
else:
    print('KITTI 원본에서 3,000장을 만들어요. 12GB 복사와 압축 해제에 시간이 걸려요...')
    !cp {KITTI_IMG_ZIP} /content/img.zip && cp {KITTI_LABEL_ZIP} /content/lab.zip
    !unzip -q -o /content/img.zip -x 'testing/*' -d /content/kitti && unzip -q -o /content/lab.zip -x 'testing/*' -d /content/kitti
    !rm /content/img.zip /content/lab.zip
    !cd {REPO} && python tools/prepare_kitti.py --kitti-root /content/kitti --out {DATA} --num-images {NUM_IMAGES}
    !cd /content && zip -qr kitti_yolo.zip kitti_yolo && cp kitti_yolo.zip {SUBSET_ZIP}
    !rm -rf /content/kitti
    print('완료: 드라이브에 kitti_yolo.zip이 생겼어요. 이제 드라이브의 data_object_image_2.zip(12GB)은 지워도 돼요.')
if not os.path.exists(f'{DATA}/images/train'):
    raise FileNotFoundError('데이터 준비가 끝나지 않았어요. 위 출력에서 빨간 오류를 확인하세요.')
print('train 이미지', len(os.listdir(f'{DATA}/images/train')), '장 / val 이미지', len(os.listdir(f'{DATA}/images/val')), '장')

In [ ]:
# 4. 학습 결과(runs)를 드라이브에 바로 저장되게 연결 — 끊겨도 결과가 남아요
assert os.path.exists(f'{REPO}/train.py'), '3번 셀을 먼저 성공시켜야 해요'
!mkdir -p {RUNS_DIR} && rm -rf {REPO}/runs && ln -s {RUNS_DIR} {REPO}/runs
%cd {REPO}
!python -m yolo26.model

In [ ]:
# 5. 학습 함수: 같은 이름의 실험이 끊겨 있으면 last.pt에서 이어서, 없으면 새로 시작
import torch

def run(name, *overrides):
    os.chdir(REPO)
    if not torch.cuda.is_available():
        raise RuntimeError('GPU가 없어요. 런타임 -> 런타임 유형 변경 -> T4 GPU로 바꾼 뒤 1번 셀부터 다시 실행하세요.')
    last = f'{REPO}/runs/train/{name}/weights/last.pt'
    ov = ' '.join(overrides)
    if os.path.exists(last):
        print(f'[{name}] 이어서 학습')
        get_ipython().system(f'python train.py --resume {last} data.root={DATA} train.workers=2')
    else:
        print(f'[{name}] 새로 학습')
        get_ipython().system(f'python train.py --cfg configs/kitti_yolo26n.yaml --name {name} data.root={DATA} train.workers=2 {ov}')

# Baseline: YOLO26-n (T4에서 약 1.5~2.5시간)
run('yolo26n')

In [ ]:
# 6. 평가: NMS-free(one-to-one) vs one-to-many + NMS
assert os.path.exists(f'{REPO}/runs/train/yolo26n/weights/best.pt'), '학습 결과(best.pt)가 없어요. 5번 학습 셀이 끝까지 돌았는지 확인하세요.'
%cd {REPO}
!python val.py --weights runs/train/yolo26n/weights/best.pt --data-root {DATA}
!python val.py --weights runs/train/yolo26n/weights/best.pt --data-root {DATA} --nms

In [ ]:
# 7. 결과 그림 몇 장 보기
assert os.path.exists(f'{REPO}/runs/train/yolo26n/weights/best.pt'), '학습 결과(best.pt)가 없어요. 5번 학습 셀이 끝까지 돌았는지 확인하세요.'
%cd {REPO}
!python predict.py --weights runs/train/yolo26n/weights/best.pt --source {DATA}/images/val --max 6
import glob
from IPython.display import Image, display
latest = sorted(glob.glob(f'{REPO}/runs/predict/exp*'), key=os.path.getmtime)[-1]
for p in sorted(glob.glob(f'{latest}/*.png'))[:6]:
    display(Image(p, width=900))

## 오답 분석 — 무엇을 어떻게 틀렸는지
혼동행렬(행 = 예측, 열 = 정답, 마지막 행·열 = 배경), 틀린 유형별 개수, 물체 크기별 recall, 가장 많이 틀린 이미지를 만들어요.
결과는 드라이브 `yolo26_runs/train/<실험 이름>/analysis/`에 저장돼요.

In [ ]:
# 8. 오답 분석 (baseline)
%cd {REPO}
from IPython.display import Image, display

def analyze(name, nms=False):
    flag = '--nms' if nms else ''
    get_ipython().system(f'python tools/analyze_errors.py --weights runs/train/{name}/weights/best.pt --data-root {DATA} {flag}')
    d = f'{REPO}/runs/train/{name}/analysis' + ('_nms' if nms else '')
    display(Image(f'{d}/confusion_matrix.png', width=600))
    display(Image(f'{d}/errors_top.png', width=1000))   # 가장 많이 틀린 8장 (전체는 analysis/errors/ 폴더)

analyze('yolo26n')

## BoTNet 비교
Baseline과 모든 설정이 같고, backbone 마지막 단계(P5)의 bottleneck 3×3 conv만 BoTNet의 relative-position MHSA로 바꾼 버전이에요.

In [ ]:
# 9. YOLO26-n + BoTNet 학습 (약 1.5~2.5시간, 끊기면 이 셀만 다시 실행)
run('yolo26n_botnet', 'model.botnet=true')

In [ ]:
# 10. Baseline vs BoTNet 비교표 (보고서에 그대로 붙여 넣을 수 있는 markdown 표) + 학습 곡선
%cd {REPO}
!python tools/compare_runs.py runs/train/yolo26n runs/train/yolo26n_botnet --data-root {DATA} --out runs/compare.md
!python tools/plot_results.py runs/train/yolo26n runs/train/yolo26n_botnet --out runs/compare.png
from IPython.display import Image, display
display(Image(f'{REPO}/runs/compare.png', width=1000))

In [ ]:
# 11. BoTNet 오답 분석 — baseline 결과(8번)와 나란히 비교해 보세요
%cd {REPO}
analyze('yolo26n_botnet')

## 작은 물체(보행자) 개선 실험 — NWD · P2 head
학습 노트(`YOLO26_NWD_P2_학습노트.pdf`)의 실험 B·C예요. baseline과 다른 설정은 모두 같아요.
- **B. NWD**: 학습 정답 고르기(TAL)와 box loss에서 CIoU 절반을 NWD로 바꿈. 모델 구조·속도는 그대로. (L4 약 1시간 40분)
- **C. P2 head**: stride 4 검출층을 하나 더 붙임. 연산 2.88 → 4.06 GFLOPs. (L4 약 2.5~3시간)

처음 실행할 때 NWD 상수 C가 자동으로 계산돼 로그에 `C = ... px`로 나와요(보행자 평균 크기).
끊기면 해당 셀만 다시 실행하면 이어서 학습해요.

In [ ]:
# 12. 실험 B: YOLO26-n + NWD
run('yolo26n_nwd', 'assign.nwd_ratio=0.5', 'loss.nwd_ratio=0.5')

In [ ]:
# 13. 실험 C: YOLO26-n + P2 head
run('yolo26n_p2', 'model.p2=true')

In [ ]:
# 14. (시간 되면) 실험 D: NWD + P2
run('yolo26n_nwd_p2', 'model.p2=true', 'assign.nwd_ratio=0.5', 'loss.nwd_ratio=0.5')

In [ ]:
# 15. 비교표 + 학습 곡선 (끝난 실험만 자동으로 포함)
%cd {REPO}
names = [n for n in ['yolo26n', 'yolo26n_botnet', 'yolo26n_nwd', 'yolo26n_p2', 'yolo26n_nwd_p2']
         if os.path.exists(f'{REPO}/runs/train/{n}/weights/best.pt')]
runs = ' '.join(f'runs/train/{n}' for n in names)
print('비교할 실험:', names)
!python tools/compare_runs.py {runs} --data-root {DATA} --out runs/compare_small.md
!python tools/plot_results.py {runs} --out runs/compare_small.png
from IPython.display import Image, display
display(Image(f'{REPO}/runs/compare_small.png', width=1000))

In [ ]:
# 16. 오답 분석 — 'Recall by box width at the network input' 표의 앞 두 칸(<8px, 8-16px)이 핵심
#     baseline도 다시 돌려야 이 새 표가 생겨요 (학습은 다시 안 함, 몇 분)
%cd {REPO}
for n in names:
    print('=' * 30, n)
    analyze(n)